[![Fixel Algorithms](https://fixelalgorithms.co/images/CCExt.png)](https://fixelalgorithms.gitlab.io)

# Deep Learning Methods

## Deep Learning - Generative Models - Conditional Variational Auto Encoder

> Notebook by:
> - Royi Avital RoyiAvital@fixelalgorithms.com

## Revision History

| Version | Date       | User        |Content / Changes                                                   |
|---------|------------|-------------|--------------------------------------------------------------------|
| 1.0.000 | 10/08/2026 | Royi Avital | First version                                                      |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FixelAlgorithmsTeam/FixelCourses/blob/master/AIProgram/2024_02/0099DeepLearningObjectDetection.ipynb)

In [ ]:
# Import Packages

# General Tools
import numpy as np
import scipy as sp
import pandas as pd

# Machine Learning

# Deep Learning
import torch
import torch.nn            as nn
import torch.nn.functional as F

import torchvision
from torchvision.transforms import v2 as TorchVisionTrns

import torchinfo
import torchvista

from torchmetrics.functional import r2_score

# Miscellaneous
import math
import os
from platform import python_version
import random

# Typing
from typing import Callable, Literal, Optional, Self, Tuple, Union
from numpy.typing import NDArray
from torch import Tensor

# Visualization
import matplotlib.pyplot as plt

# Jupyter
from IPython import get_ipython

## Notations

* <font color='red'>(**?**)</font> Question to answer interactively.
* <font color='blue'>(**!**)</font> Simple task to add code for the notebook.
* <font color='green'>(**@**)</font> Optional / Extra self practice.
* <font color='brown'>(**#**)</font> Note / Useful resource / Food for thought.

Code Notations:

```python
someVar    = 2; #<! Notation for a variable
vVector    = np.random.rand(4) #<! Notation for 1D array
mMatrix    = np.random.rand(4, 3) #<! Notation for 2D array
tTensor    = np.random.rand(4, 3, 2, 3) #<! Notation for nD array (Tensor)
tuTuple    = (1, 2, 3) #<! Notation for a tuple
lList      = [1, 2, 3] #<! Notation for a list
dDict      = {1: 3, 2: 2, 3: 1} #<! Notation for a dictionary
oObj       = MyClass() #<! Notation for an object
dfData     = pd.DataFrame() #<! Notation for a data frame
dsData     = pd.Series() #<! Notation for a series
hObj       = plt.Axes() #<! Notation for an object / handler / function handler
```

### Code Exercise

 - Single line fill

```python
valToFill = ???
```

 - Multi Line to Fill (At least one)

```python
# You need to start writing
?????
```

 - Section to Fill

```python
#===========================Fill This===========================#
# 1. Explanation about what to do.
# !! Remarks to follow / take under consideration.
mX = ???

?????
#===============================================================#
```

In [ ]:
# Configuration
# %matplotlib inline

seedNum = 512
np.random.seed(seedNum)
random.seed(seedNum)

# Matplotlib default color palette
lMatPltLibclr = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf']
# sns.set_theme() #>! Apply SeaBorn theme

runInGoogleColab = 'google.colab' in str(get_ipython())

# Improve performance by benchmarking
torch.backends.cudnn.benchmark = True

# Reproducibility (Per PyTorch Version on the same device)
# torch.manual_seed(seedNum)
# torch.backends.cudnn.deterministic = True
# torch.backends.cudnn.benchmark     = False #<! Makes things slower

In [ ]:
# Constants

FIG_SIZE_DEF    = (8, 8)
ELM_SIZE_DEF    = 50
CLASS_COLOR     = ('b', 'r')
EDGE_COLOR      = 'k'
MARKER_SIZE_DEF = 10
LINE_WIDTH_DEF  = 2

D_CLASSES   = {ii: str(ii) for ii in range(10)}
L_CLASSES   = [str(ii) for ii in range(10)]
TU_IMG_SIZE = (28, 28, 1)

PROJECT_NAME       = 'FixelCourses'
DATA_FOLDER_NAME   = 'DataSets'
MODELS_FOLDER_NAME = 'Models'
BASE_FOLDER_PATH   = os.getcwd()[:(len(os.getcwd()) - (os.getcwd()[::-1].lower().find(PROJECT_NAME.lower()[::-1])))]
DATA_FOLDER_PATH   = os.path.join(BASE_FOLDER_PATH, DATA_FOLDER_NAME)
MODELS_FOLDER_PATH = os.path.join(BASE_FOLDER_PATH, MODELS_FOLDER_NAME)

TENSOR_BOARD_BASE = 'TB'

D_CLASSES = {ii: str(ii) for ii in range(10)}
L_CLASSES = [ii for ii in range(len(D_CLASSES))]

In [ ]:
# Download Auxiliary Modules for Google Colab
if runInGoogleColab:
    !wget https://raw.githubusercontent.com/FixelAlgorithmsTeam/FixelCourses/master/AIProgram/2024_02/DataManipulation.py
    !wget https://raw.githubusercontent.com/FixelAlgorithmsTeam/FixelCourses/master/AIProgram/2024_02/DataVisualization.py
    !wget https://raw.githubusercontent.com/FixelAlgorithmsTeam/FixelCourses/master/AIProgram/2024_02/DeepLearningPyTorch.py

In [ ]:
# Courses Packages

from DataManipulation import DownloadUrl
from DataVisualization import AnnotateImage, PlotLabelsHistogram, PlotMnistImages, PlotScatterData
from DeepLearningPyTorch import TrainModel

In [ ]:
# General Auxiliary Functions

def TensorImageNumpy( tZ: Tensor ) -> NDArray:
    """
    Converts a PyTorch Tensor to a Numpy Array.
    """
    mZ = tZ.squeeze()
    mX = mZ.detach().cpu().numpy()

    return mX

class MNISTDatasetCSV(torch.utils.data.Dataset):
    """
    MNIST Dataset from CSV File.  
    Supports Image Classification and Self Supervised Learning tasks.
    """

    def __init__( self: Self, csvFilePath: str, subSetType: Literal['All', 'Train', 'Val'], *, tuImgSize: Tuple[int, ...] = (28, 28), hImgTrns: Optional[Callable] = None, hFeatTrns: Optional[Callable] = None, hTgtTrns: Optional[Callable] = None ) -> None:
        """
        Constructor Method.

        Parameters
        ----------
        csvFilePath : str
            Path / URL to the CSV file.
        subSetType : Literal['All', 'Train', 'Val']
            Subset type: 'All' for the entire dataset, 'Train' for training set, 'Val' for validation set
        tuImgSize : Tuple[int, ...], optional
            Image size tuple, by default (28, 28)
        imgTrns : Optional[Callable], optional
            Transform to be applied to the images, by default None
        """

        dfData = pd.read_csv(csvFilePath)
        match subSetType:
            case 'All':
                pass
            case 'Train':
                dfData = dfData.iloc[:60000, :].reset_index(drop = True)
            case 'Val':
                dfData = dfData.iloc[60000:, :].reset_index(drop = True)
            case _:
                raise ValueError(f'Unsupported subset type: {subSetType}')
        
        dsLbl = dfData.iloc[:, -1]
        dsLbl = dsLbl.astype(np.uint8)

        # Convert data to NumPy arrays for better performance
        mData = dfData.iloc[:, :-1].to_numpy(np.uint8, copy = True)
        vLbl  = dfData.iloc[:, -1].to_numpy(np.uint8, copy = True)

        self._dfData    = dfData
        self._mData     = mData
        self._dsLbl     = dsLbl
        self._vLbl      = vLbl
        self._tuImgSize = tuImgSize
        self._hImgTrns  = hImgTrns
        self._hTgtTrns  = hTgtTrns
        self._hFeatTrns = hFeatTrns

    def __len__( self: Self ) -> int:
        """
        Returns the number of samples in the dataset.
        """
        return len(self._dfData)

    def __getitem__( self: Self, idx: int ) -> Tuple[Tensor, Tuple[Union[int, Tensor], Tensor]]:
        """
        Returns the sample at the given index.

        Parameters
        ----------
        idx : int
            Index of the sample to be retrieved.

        Returns
        -------
        Tuple[torch.Tensor, Tuple[Union[int, torch.Tensor], torch.Tensor]]
            Tuple containing the sample tensor and a tuple of the label and the target tensor.
        """
        
        tX    = self._mData[idx]
        tX    = np.reshape(tX, self._tuImgSize)
        valY  = self._vLbl[idx]

        if self._hImgTrns:
            # Assuming Torchvision v2 transforms
            tX = self._hImgTrns(tX)

        # Create a copy for feature transform
        # Handle the case of NumPy Array and Torch Tensor
        if isinstance(tX, np.ndarray):
            tY = tX.copy()
        else:
            tY = tX.clone()
        
        if self._hFeatTrns:
            tX = self._hFeatTrns(tX)
        
        if self._hTgtTrns:
            tY = self._hTgtTrns(tY)

        return tX, (valY, tY)
    
    def GetLabels( self: Self ) -> NDArray:
        """
        Returns all labels in the dataset.

        Returns
        -------
        NDArray
            Array of labels.
        """
        return self._dsLbl.to_numpy()
    
    def SetTransform( self: Self, trnsType: Literal['Feature', 'Image', 'Target'], hTrns: Optional[Callable] ) -> None:
        """
        Sets the transform for the dataset.

        Parameters
        ----------
        trnsType : Literal['Feature', 'Image', 'Target']
            Type of transform to be set.
        hTrns : Optional[Callable]
            Transform to be applied.
        """

        match trnsType:
            case 'Feature':
                self._hFeatTrns = hTrns
            case 'Image':
                self._hImgTrns = hTrns
            case 'Target':
                self._hTgtTrns = hTrns
            case _:
                raise ValueError(f'Unsupported transform type: {trnsType}')

class ConditionalInput(list):
    # PyTorch's collate function converts tuples to lists.
    # This handles the `tX.shape[0]` in `TrainModel` -> `RunEpoch`
    @property
    def shape( self: Self ) -> torch.Size:
        return self[0].shape

# A class to wrap the dataset and provide conditional input for the model to have both `tX` and `valY` as input
class ConditionalDataset(torch.utils.data.Dataset):
    def __init__( self: Self, dsData: torch.utils.data.Dataset ) -> None:
        self.dsData = dsData

    def __len__( self: Self ) -> int:
        return len(self.dsData)

    def __getitem__( self: Self, idx: int ) -> Tuple[ConditionalInput, Tuple[int, Tensor]]:
        tX, (valY, tY) = self.dsData[idx]
        valY = int(valY)

        return ConditionalInput((tX, valY)), (valY, tY)

* <font color='blue'>(**!**)</font> Go through `MNISTDatasetCSV` class. Understand how it serves the _Self Supervised_ concept.

## Conditional Variational Auto Encoder

![](https://i.imgur.com/ylhM6C7.png)
<!-- ![](https://i.postimg.cc/QN4yMnKL/Diagrams-Conditional-Variational-Auto-Encoder.png) -->

A _Conditional Variational Auto Encoder_ (CVAE) is a generative model which learns a structured latent space while controlling the generated sample using additional information ${\color{cyan}{\boldsymbol{y}}}$.  
For MNIST, ${\color{cyan}{\boldsymbol{y}}}$ is the digit label, commonly represented by a _One Hot Encoded_ vector.

The model is composed of:
 - _Encoder_: Transforms the input $\color{cyan}{\boldsymbol{x}}$ together with the condition ${\color{cyan}{\boldsymbol{y}}}$ into the parameters of a distribution: a mean $\boldsymbol{\mu} \left( {\color{cyan}{\boldsymbol{x}}}, {{\color{cyan}{\boldsymbol{y}}}} \right)$ and a vector of standard deviations $\boldsymbol{\sigma} \left( {\color{cyan}{\boldsymbol{x}}}, {{\color{cyan}{\boldsymbol{y}}}} \right)$.  
 - _Embedding_: Samples $\color{green}{\boldsymbol{z}}$ from the encoded distribution using the _Reparameterization Trick_: $\color{green}{\boldsymbol{z}} = \boldsymbol{\mu} + \boldsymbol{\sigma} \odot \boldsymbol{\epsilon}$, where $\boldsymbol{\epsilon} \sim \mathcal{N} \left( \boldsymbol{0}, \boldsymbol{I} \right)$.  
 - _Decoder_: Reconstructs the input from both $\color{green}{\boldsymbol{z}}$ and the condition ${\color{cyan}{\boldsymbol{y}}}$.  

The condition tells the decoder _what_ to generate, while $\color{green}{\boldsymbol{z}}$ controls variations within the selected class.  
For example, fixing ${\color{cyan}{\boldsymbol{y}}}$ to the digit $7$ and sampling different $\color{green}{\boldsymbol{z}}$ values generates different styles of the digit $7$.

The CVAE loss is composed of:
 - _Reconstruction Loss_  
   Encourages the decoded sample to match the input under the given condition.
 - _Regularization Loss_  
   Uses the KL Divergence to make the encoded distribution close to the prior $\mathcal{N} \left( \boldsymbol{0}, \boldsymbol{I} \right)$.

Some use cases of _Conditional Variational Auto Encoders_:

 - Controlled Data Generation  
   Select ${\color{cyan}{\boldsymbol{y}}}$, sample $\color{green}{\boldsymbol{z}} \sim \mathcal{N} \left( \boldsymbol{0}, \boldsymbol{I} \right)$ and pass both through the decoder.
 - Class Conditional Interpolation  
   Fix ${\color{cyan}{\boldsymbol{y}}}$ and move smoothly in the latent space to generate smooth variations within the selected class.
 - Attribute Controlled Generation  
   Use attributes, measurements or other side information as the condition.

</br> 

* <font color='brown'>(**#**)</font> A regular VAE must encode both the class and the style in $\color{green}{\boldsymbol{z}}$.  
  A CVAE provides the class explicitly, hence $\color{green}{\boldsymbol{z}}$ can focus on variations within the class.
* <font color='brown'>(**#**)</font> During generation, the condition must be supplied even though no input image is available.
* <font color='brown'>(**#**)</font> The reconstruction and regularization terms still create a tradeoff between accurate reconstruction and a smooth latent space.

This notebook demonstrates:
 - Conditioning the _Encoder_ on the image and its label.
 - Applying the _Reparameterization Trick_ to enable gradient based training.
 - Conditioning the _Decoder_ on the latent vector and the label.
 - Training the CVAE using reconstruction and KL Divergence losses.
 - Selecting a class and sampling from the prior to generate new images.
 - Exploring variations within each class in the latent space.

</br>

* <font color='brown'>(**#**)</font> The condition does not have to be a class label. It may be any useful information available during training and generation.

In [ ]:
# Parameters

# Data
csvFileName = 'MNIST.csv'
csvFileUrl  = r'https://huggingface.co/datasets/Royi/MNIST/resolve/main/MNIST.csv'

tuImgSize = (TU_IMG_SIZE[0], TU_IMG_SIZE[1]) #<! Image spatial size
numCls    = len(L_CLASSES) #<! Number of classes

# Model
modelName   = 'ModelConditionalVariationalAutoEncoder_2026_08_10.pt' #<! OneDrive -> Courses -> Models -> AIProgram
decEmbedDim = 4
latDim      = 2

# Loss
recLossType = 'MSE'
β           = 1.0

# Training
batchSize   = 512
numWorkers  = 2 #<! Number of workers
numEpochs   = 45

# Visualization
numImg = 3

## Generate / Load Data

The data is the MNIST Dataset.  
This section:

 - Defines the `Dataset` class.  
   It should support the case the labels are the image itself.
 - Create a _Train_ and _Validation_ datasets.
 - Plot the data.
 - Define the _Augmentation_ / _Transform_.
 - Define the `Dataloader`.

In [ ]:
# Download Data (CSV)

csvFilePath = os.path.join(DATA_FOLDER_PATH, csvFileName)
csvFilePath = DownloadUrl(csvFileUrl, csvFilePath)

In [ ]:
# Data Set

dsTrain = MNISTDatasetCSV(csvFilePath, 'Train')
dsVal   = MNISTDatasetCSV(csvFilePath, 'Val')

print(f'The number of samples in training data set  : {len(dsTrain)}')
print(f'The number of samples in validation data set: {len(dsVal)}')

In [ ]:
# Element of the Data Set / Data Sample

tX, tuY = dsTrain[0]
valY    = tuY[0]
tY      = tuY[1]

print(f'The features shape: {tX.shape}')
print(f'The target shape  : {tY.shape}')
print(f'The label         : {valY}')

### Plot the Data

In [ ]:
# Plot the Data

mX = np.zeros((9, 28 * 28), dtype = np.uint8)
vY = np.zeros((9,), dtype = np.uint8)

for ii in range(9):
    randIdx = random.randint(0, len(dsTrain) - 1)
    tX, tuY = dsTrain[randIdx]
    valY    = tuY[0]
    mX[ii]  = tX.flatten()
    vY[ii]  = valY

hF = PlotMnistImages(mX, vY, 3, 3)

In [ ]:
# Plot Single Sample

randIdx = random.randint(0, len(dsTrain) - 1)
tX, tuY = dsTrain[randIdx]
valY    = tuY[0]

hF, hA = plt.subplots(figsize = (7, 7))
hA.imshow(tX, cmap = 'gray')
hA.set_title(f'Sample Index: {randIdx}, Label: {valY}')
AnnotateImage(tX, hA, fontSize = 6);

In [ ]:
# Histogram of Labels

hF, vHa = plt.subplots(nrows = 1, ncols = 2, figsize = (8, 4))
vHa = vHa.flat

hA = PlotLabelsHistogram(dsTrain.GetLabels(), hA = vHa[0], lClass = L_CLASSES)
hA.set_title('Histogram of Labels, Training Set');

hA = PlotLabelsHistogram(dsVal.GetLabels(), hA = vHa[1], lClass = L_CLASSES)
hA.set_title('Histogram of Labels, Validation Set');

### Augmentation / Transform

In [ ]:
# Loader Transform

oTrns = TorchVisionTrns.Compose([
    TorchVisionTrns.ToImage(),
    TorchVisionTrns.ToDtype(torch.float, scale = True),
])

In [ ]:
# Apply Transforms

dsTrain.SetTransform('Image', oTrns)
dsVal.SetTransform('Image', oTrns)

In [ ]:
# Element of the Data Set / Data Sample

tX, tuY = dsTrain[0]
valY    = tuY[0]
tY      = tuY[1]

print(f'The features shape: {tX.shape}')
print(f'The target shape  : {tY.shape}')
print(f'The label         : {valY}')

In [ ]:
# Plot Single Sample

randIdx = random.randint(0, len(dsTrain) - 1)
tX, tuY = dsTrain[randIdx]
valY    = tuY[0]

mX = TensorImageNumpy(tX)

hF, hA = plt.subplots(figsize = (4, 4))
hA.imshow(mX, cmap = 'gray')
hA.set_title(f'Sample Index: {randIdx}, Label: {valY}');

* <font color='red'>(**?**)</font> What other augmentation would work for this case? Should flips be used?

### Data Loaders

In [ ]:
# Data Loader

dlTrain = torch.utils.data.DataLoader(ConditionalDataset(dsTrain), batch_size = batchSize, shuffle = True, num_workers = 0, pin_memory = torch.cuda.is_available())
dlVal   = torch.utils.data.DataLoader(ConditionalDataset(dsVal), batch_size = 2 * batchSize, shuffle = False, num_workers = 0, pin_memory = torch.cuda.is_available())

In [ ]:
# Iterate on the Loader
# The first batch.
tX, tuY = next(iter(dlTrain)) #<! PyTorch Tensors

print(f'The batch features dimensions: {tX.shape}')
print(f'The batch labels dimensions: {tuY[0].shape}')
print(f'The batch targets dimensions: {tuY[1].shape}')

## Build Conditional Variational Auto Encoder Model

In a _Conditional Variational Auto Encoder_ the encoder does not map each sample into a single point.  
It maps the sample ${\color{cyan}{\boldsymbol{x}}}_{i}$ together with its condition ${\color{cyan}{\boldsymbol{y}}}_{i}$ into a distribution in the latent space:

$$ q_{\boldsymbol{w}} \left( \boldsymbol{z} \mid {\color{cyan}{\boldsymbol{x}}}_{i}, {\color{cyan}{\boldsymbol{y}}}_{i} \right) = \mathcal{N} \left( \boldsymbol{\mu}_{i}, \operatorname{Diag} \left( \boldsymbol{\sigma}_{i}^{2} \right) \right) $$

A latent vector is sampled using the _Reparameterization Trick_:

$$ {\color{green}{\boldsymbol{z}}}_{i} = \boldsymbol{\mu}_{i} + \boldsymbol{\sigma}_{i} \odot \boldsymbol{\epsilon}, \qquad \boldsymbol{\epsilon} \sim \mathcal{N} \left( \boldsymbol{0}, \boldsymbol{I} \right) $$

This form separates the random sampling from $\boldsymbol{\mu}_{i}$ and $\boldsymbol{\sigma}_{i}$, hence gradients can propagate through the encoder.  
The decoder receives both ${\color{green}{\boldsymbol{z}}}_{i}$ and ${\color{cyan}{\boldsymbol{y}}}_{i}$, hence the reconstruction is given by $\phi_{\boldsymbol{w}} \left( {\color{green}{\boldsymbol{z}}}_{i}, {\color{cyan}{\boldsymbol{y}}}_{i} \right)$.

The CVAE objective can be written as:

$$ \arg \min_{\boldsymbol{w}} \sum_{i} \left[ \mathbb{E}_{q_{\boldsymbol{w}} \left( \boldsymbol{z} \mid {\color{cyan}{\boldsymbol{x}}}_{i}, {\color{cyan}{\boldsymbol{y}}}_{i} \right)} \left[ \mathcal{L}_{\mathrm{Rec}} \left( \phi_{\boldsymbol{w}} \left( \boldsymbol{z}, {\color{cyan}{\boldsymbol{y}}}_{i} \right), {\color{cyan}{\boldsymbol{x}}}_{i} \right) \right] + \beta D_{\mathrm{KL}} \left( q_{\boldsymbol{w}} \left( \boldsymbol{z} \mid {\color{cyan}{\boldsymbol{x}}}_{i}, {\color{cyan}{\boldsymbol{y}}}_{i} \right) \,\|\, \mathcal{N} \left( \boldsymbol{0}, \boldsymbol{I} \right) \right) \right] $$

Where $\phi_{\boldsymbol{w}}$ is the conditional decoder and $\beta$ controls the regularization strength.

The objective is composed of:
 - _Reconstruction Loss_  
   Makes the decoded sample under the given condition similar to the input.  
   In case of Gaussian Noise it matches: $\frac{1}{N} \sum_{i = 1}^{N} {\left\| {\color{cyan}{\boldsymbol{x}}}_{i} - \phi_{\boldsymbol{w}} \left( {\color{green}{\boldsymbol{z}}}_{i}, {\color{cyan}{\boldsymbol{y}}}_{i} \right) \right\|}_{2}^{2}$
 - _KL Divergence Loss_  
   Makes each encoded distribution similar to the prior $\mathcal{N} \left( \boldsymbol{0}, \boldsymbol{I} \right)$.  
   This organizes the latent space and makes conditional sampling from the prior meaningful.  
   In the case above, for a diagonal covariance, it sums to: $D_{\mathrm{KL}} \left( \mathcal{N}_{d} \left( \boldsymbol{\mu}, \operatorname{Diag} \left( \boldsymbol{\sigma}^{2} \right) \right) \,\|\, \mathcal{N}_{d} \left( \boldsymbol{0}, \boldsymbol{I} \right) \right) = \frac{1}{2} \sum_{j = 1}^{d} \left( \sigma_{j}^{2} + \mu_{j}^{2} - 1 - \log \left( \sigma_{j}^{2} \right) \right)$.

Hence the overall Loss Function is given by:

$$ \mathcal{L}_{\mathrm{CVAE}} = \frac{1}{N} \sum_{i = 1}^{N} \left[ {\left\| {\color{cyan}{\boldsymbol{x}}}_{i} - \phi_{\boldsymbol{w}} \left( {\color{green}{\boldsymbol{z}}}_{i}, {\color{cyan}{\boldsymbol{y}}}_{i} \right) \right\|}_{2}^{2} + \frac{\beta}{2} \sum_{j = 1}^{d} \left( \sigma_{i,j}^{2} + \mu_{i,j}^{2} - 1 - \log \left( \sigma_{i,j}^{2} \right) \right) \right] $$

</br>

* <font color='brown'>(**#**)</font> A small $\beta$ favors accurate reconstruction but may create a less organized latent space.  
  At the extreme $\beta = 0$, it is equivalent to a _Conditional Auto Encoder_.
* <font color='brown'>(**#**)</font> A large $\beta$ creates stronger regularization but may remove useful information from the latent representation.  
* <font color='brown'>(**#**)</font> During training, the expectation is usually approximated using a single sample of $\boldsymbol{\epsilon}$ for each input.
* <font color='brown'>(**#**)</font> During generation, sample $\boldsymbol{z} \sim \mathcal{N} \left( \boldsymbol{0}, \boldsymbol{I} \right)$ and provide the desired condition $\boldsymbol{y}$ to the decoder.
* <font color='brown'>(**#**)</font> The encoder commonly predicts $\log \left( \boldsymbol{\sigma}^{2} \right)$ instead of $\boldsymbol{\sigma}$ for numerical stability.

### Embedding ${\color{cyan}{\boldsymbol{y}}}$ into the Encoder and Decoder

There are many ways to utilize the information in ${\color{cyan}{\boldsymbol{y}}}$ given its categorical nature:

 - _One Hot Encoding_  
   Concatenate the one hot vector directly to the input or to an intermediate feature vector.
 - _Learned Embedding_  
   Map each category into a learned dense vector and concatenate it with the model features.
 - _Conditional Normalization_  
   Use the category to control the scale and bias parameters of normalization layers.
 - _Feature Modulation_  
   Use the category to add, multiply or gate intermediate feature maps.

This notebook utilizes the _Learned Embedding_ method with PyTorch's [`nn.Embedding`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Embedding.html).  
The method represents a categorical value by a _One Hot_ vector and transforms it into an embedding space using a _Linear Layer_.  
Let $\boldsymbol{E} \in \mathbb{R}^{d \times K}$ be an embedding matrix for $K$ categories, where each category is mapped into a vector of dimension $d$.  
For category $y$, let $\boldsymbol{e}_{y} \in \mathbb{R}^{K}$ be its one hot representation. The embedded vector is:

$$ \boldsymbol{v}_{y} = \boldsymbol{E} \boldsymbol{e}_{y} = \boldsymbol{E} \left[ y, : \right] $$

Due to the special form of $\boldsymbol{e}_{y}$, the multiplication only selects row $y$ of $\boldsymbol{E}$.  
Hence `nn.Embedding` receives the category index directly and performs the lookup without forming the one hot vector.

The model applies the embedding trick twice:

 - _Encoder_: The embedding dimension matches the number of pixels in the input image.  
   The embedding is reshaped as an image and concatenated with the input as an additional channel.
 - _Decoder_: The embedding dimension is arbitrary.  
   The embedding is concatenated with the sampled vector ${\color{green}{\boldsymbol{z}}}$.

</br>

* <font color='brown'>(**#**)</font> The two embedding layers do not have to share their weights or embedding dimensions.  
  Each one learns the representation most useful for its location in the model.

In [ ]:
# Sampling Layer

class GaussianSamplingLayer(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, mμ: Tensor, mLogΣ: Tensor) -> Tensor:
        """
        Args:
            mμ   : Tensor of mean values (batchSize, latDim)
            mLogΣ: Tensor of log variances (batchSize, latDim)
        
        Note: Using `mLogΣ` instead of `mσ` for numerical stability.
        """

        mσ = torch.exp(0.5 * mLogΣ) #<! The standard deviation from log variance
        mε = torch.randn_like(mσ) #<! Sample random noise from standard normal distribution
        mZ = mμ + mε * mσ #<! Reparameterization trick: z = μ + σ * ε, where ε ~ N(0, I)

        return mZ

In [ ]:
# Encoder Decoder Model

class ConditionalVariationalAutoEncoder(nn.Module):
    def __init__(self, numCls: int, tuImgSize: Tuple[int, int], latDim: int, decEmbedDim: int) -> None:
        super().__init__()

        outDim = 2 * latDim
        decDim = latDim + decEmbedDim

        # Encoder: 1x28x28 -> `outDim`
        self.oEncEmbedding = nn.Embedding(num_embeddings = numCls, embedding_dim = tuImgSize[0] * tuImgSize[1]) #<! Embedding layer for class labels
        self.oEnc = nn.Sequential(
            nn.Conv2d(2,  8,      kernel_size = 5, bias = False),             nn.BatchNorm2d(8 ), nn.LeakyReLU(), #<! (2, 28, 28) -> (8, 24, 24)
            nn.Conv2d(8,  16,     kernel_size = 5, bias = False),             nn.BatchNorm2d(16), nn.LeakyReLU(), #<! (8, 24, 24) -> (16, 20, 20)
            nn.Conv2d(16, 32,     kernel_size = 5, bias = False, stride = 2), nn.BatchNorm2d(32), nn.LeakyReLU(), #<! (16, 20, 20) -> (32, 8, 8)
            nn.Conv2d(32, 64,     kernel_size = 5, bias = False),             nn.BatchNorm2d(64), nn.LeakyReLU(), #<! (32, 8, 8) -> (64, 4, 4)
            nn.Conv2d(64, outDim, kernel_size = 4),                                                               #<! (64, 4, 4) -> (outDim, 1, 1)
            nn.Flatten()                                                                                          #<! (outDim, 1, 1) -> (outDim,)
        )

        # Samples: (outDim, ) -> (latDim, ), (latDim, ) -> (latDim, )
        self.oSampler = GaussianSamplingLayer()

        # Decoder: `latDim` -> 1x28x28
        self.oDecEmbedding = nn.Embedding(num_embeddings = numCls, embedding_dim = decEmbedDim) #<! Embedding layer for class labels
        self.oDec = nn.Sequential(
            nn.Unflatten(dim = 1, unflattened_size = (decDim, 1, 1)), #<! Reshape (N, latDim + decEmbedDim) -> (N, latDim + decEmbedDim, 1, 1)
            nn.Upsample(scale_factor = 2), nn.Conv2d(decDim, 64, kernel_size = 3, padding = 1, bias = False), nn.BatchNorm2d(64), nn.LeakyReLU(), 
            nn.Upsample(scale_factor = 2), nn.Conv2d(64,     32, kernel_size = 3, padding = 1, bias = False), nn.BatchNorm2d(32), nn.LeakyReLU(), 
            nn.Upsample(scale_factor = 2), nn.Conv2d(32,     16, kernel_size = 3, padding = 1, bias = False), nn.BatchNorm2d(16), nn.LeakyReLU(), 
            nn.Upsample(scale_factor = 2), nn.Conv2d(16,     8,  kernel_size = 3, padding = 1, bias = False), nn.BatchNorm2d(8 ), nn.LeakyReLU(), 
            nn.Upsample(scale_factor = 2), nn.Conv2d(8,      4,  kernel_size = 3, padding = 1, bias = True ),                     nn.LeakyReLU(), 
                                           nn.Conv2d(4,      1,  kernel_size = 5, padding = 0, bias = True ),
        )

    def forward(self, tX: Union[Tensor, ConditionalInput], vY: Optional[Tensor] = None) -> Tuple[Tensor, Tensor, Tensor]:

        if vY is None:
            tX, vY = tX #<! Structured input from `ConditionalDataset`

        # Encode
        tX = torch.cat((tX, self.oEncEmbedding(vY).view(-1, 1, *tuImgSize)), dim = 1) #<! Concatenate the image and the label embedding
        mμ, mLogΣ = self.oEnc(tX).chunk(2, dim = 1) #<! Distribution Parameters

        # Sample
        if self.training:
            mZ = self.oSampler(mμ, mLogΣ) #<! Sample from the distribution
        else:
            mZ = mμ #<! Deterministic output during evaluation (Mean of the distribution)

        # Decode
        mZ = torch.cat((mZ, self.oDecEmbedding(vY)), dim = 1) #<! Concatenate the latent vector and the label embedding
        tXHat = self.oDec(mZ) #<! Reconstructed Image

        return tXHat, mμ, mLogΣ

    def GetEmbedding(self, tX: Tensor, vY: Tensor) -> Tensor:

        tX    = torch.cat((tX, self.oEncEmbedding(vY).view(-1, 1, *tuImgSize)), dim = 1) #<! Concatenate the image and the label embedding
        mμ, _ = self.oEnc(tX).chunk(2, dim = 1) #<! The mean is the deterministic embedding

        return mμ

In [ ]:
# The Model Object

oModel = ConditionalVariationalAutoEncoder(numCls, tuImgSize, latDim, decEmbedDim)

In [ ]:
# Model Summary

tX = torch.randn(batchSize, *(TU_IMG_SIZE[::-1]))
vY = torch.zeros(batchSize, dtype = torch.long)

torchinfo.summary(oModel, input_data = (tX, vY), col_names = ['kernel_size', 'input_size', 'output_size', 'num_params'], device = 'cpu', row_settings = ['depth', 'var_names'])

In [ ]:
# Model Graph

torchvista.trace_model(oModel.eval(), (tX, vY))

## Train the Model

This section defines:
 - Loss Class (_VAE Loss_) 
   Composed of 2 loses:
   - The Reconstruction Loss  
     Based on the [MSE](https://en.wikipedia.org/wiki/Mean_squared_error) or [MAE](https://en.wikipedia.org/wiki/Mean_absolute_error) loss.  
     Drives reconstruction of the image at output.
   - The Prior Loss
     Based on the [Kullback Leibler Divergence](https://en.wikipedia.org/wiki/Kullback%E2%80%93Leibler_divergence) Loss.  
     Promotes global structure of the latent space to match Normal Distribution.
 - Score  
   The R2 score.

In [ ]:
# Check GPU Availability

runDevice = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu') #<! The 1st CUDA device
print(f'The run device: {runDevice}')

In [ ]:
# The Loss Class
class ConditionalVariationalAutoEncoderLoss(nn.Module):
    def __init__( self, recLossType: Literal['MAE', 'MSE'], β: float ) -> None:
        super().__init__()

        # Both loss terms are summed per sample and averaged over the batch.
        match recLossType:
            case 'MAE':
                self.oRecLoss = nn.L1Loss(reduction = 'sum')
            case 'MSE':
                self.oRecLoss = nn.MSELoss(reduction = 'sum')
            case _:
                raise ValueError(f'Unsupported loss type: {recLossType}')
        
        self.β = β
    
    def forward( self, tuYHat: Tuple[Tensor, Tensor, Tensor], tuY: Tuple[Tensor, Tensor] ) -> Tensor:

        tXHat, mμ, mLogΣ = tuYHat
        _,      tX        = tuY

        batchSize = tX.size(0)
        
        recLoss = self.oRecLoss(tXHat, tX)
        klLoss  = 0.5 * torch.sum(mLogΣ.exp() + mμ.square() - 1.0 - mLogΣ)

        return (recLoss + self.β * klLoss) / batchSize

* <font color='red'>(**?**)</font> How does $\beta$ affect the reconstruction $R^{2}$ score and the structure of the latent space? Compare the training and validation scores.

In [ ]:
# The Score Class
class ConditionalAutoEncoderScore(nn.Module):
    def __init__( self ) -> None:
        super().__init__()
    
    def forward( self, tuYHat: Tuple[Tensor, Tensor, Tensor], tuY: Tuple[Tensor, Tensor] ) -> Tensor:

        tXHat, _, _ = tuYHat
        _, tX       = tuY
        
        r2Score = r2_score(tXHat.view(-1), tX.view(-1))
        
        return r2Score

In [ ]:
# Loss and Score
hL = ConditionalVariationalAutoEncoderLoss(recLossType, β)
hS = ConditionalAutoEncoderScore()
hL = hL.to(runDevice) #<! Not required!
hS = hS.to(runDevice)

In [ ]:
# Training the Model
oModel = oModel.to(runDevice) #<! Transfer model to device
oOpt = torch.optim.AdamW(oModel.parameters(), lr = 6e-4, betas = (0.9, 0.99), weight_decay = 1e-3) #<! Define optimizer
oSch = torch.optim.lr_scheduler.OneCycleLR(oOpt, max_lr = 2e-3, total_steps = numEpochs)
oModel, lTrainLoss, lTrainScore, lValLoss, lValScore, lLearnRate = TrainModel(oModel, dlTrain, dlVal, oOpt, numEpochs, hL, hS, oSch = oSch)

In [ ]:
# Load the Model
modelPath = os.path.join(MODELS_FOLDER_PATH, modelName)
if os.path.isfile(modelPath):
    dModel = torch.load(modelPath, map_location = runDevice)
    oModel.load_state_dict(dModel['Model'])
    print(f'Model loaded from: {modelPath}')
    oModel = oModel.to(runDevice) #<! Transfer model to device

In [ ]:
# Plot Training Phase
# Requires the variables: `lTrainLoss`, `lTrainScore`, `lValLoss`, `lValScore`, `lLearnRate` from training phase.

hF, vHa = plt.subplots(nrows = 1, ncols = 3, figsize = (15, 5))
vHa = np.ravel(vHa)

hA = vHa[0]
hA.plot(lTrainLoss, lw = 2, label = 'Train')
hA.plot(lValLoss, lw = 2, label = 'Validation')
hA.set_title(f'CVAE Loss ({recLossType}, β = {β})')
hA.set_xlabel('Epoch')
hA.set_ylabel('Loss')
hA.legend()

hA = vHa[1]
hA.plot(lTrainScore, lw = 2, label = 'Train')
hA.plot(lValScore, lw = 2, label = 'Validation')
hA.set_title('CVAE Reconstruction Score')
hA.set_xlabel('Epoch')
hA.set_ylabel('Score')
hA.legend()

hA = vHa[2]
hA.plot(lLearnRate, lw = 2)
hA.set_title('Learn Rate Scheduler')
hA.set_xlabel('Epoch')
hA.set_ylabel('Learn Rate');

* <font color='red'>(**?**)</font> How come the validation results are better? Think about the sampling process.

In [ ]:
# Inference Mode

oModel = oModel.eval()

In [ ]:
# Sample from Train
tX, (valY, _) = dsTrain[7]

tX = tX.to(runDevice).unsqueeze(0)
vY = torch.tensor([valY], dtype = torch.long, device = runDevice)

with torch.inference_mode():
    tXHat, _, _ = oModel(tX, vY)

mXHat = TensorImageNumpy(tXHat)
mX    = TensorImageNumpy(tX)

hF, vHa = plt.subplots(nrows = 1, ncols = 2, figsize = (6, 3))
vHa = vHa.flat

hA = vHa[0]
hA.imshow(mX, cmap = 'gray')
hA.set_title(f'Input Image, Label: {valY}')
hA = vHa[1]
hA.imshow(mXHat, cmap = 'gray')
hA.set_title('Conditional Reconstruction');

In [ ]:
# Sample from Validation
tX, (valY, _) = dsVal[23]

tX = tX.to(runDevice).unsqueeze(0)
vY = torch.tensor([valY], dtype = torch.long, device = runDevice)

with torch.inference_mode():
    tXHat, _, _ = oModel(tX, vY)

mXHat = TensorImageNumpy(tXHat)
mX    = TensorImageNumpy(tX)

hF, vHa = plt.subplots(nrows = 1, ncols = 2, figsize = (6, 3))
vHa = vHa.flat

hA = vHa[0]
hA.imshow(mX, cmap = 'gray')
hA.set_title(f'Input Image, Label: {valY}')
hA = vHa[1]
hA.imshow(mXHat, cmap = 'gray')
hA.set_title('Conditional Reconstruction');

In [ ]:
# Conditional Encoding

lEnc = []
lY   = []

for tuX, (vY, _) in dlTrain:
    tX, vY = tuX
    tX = tX.to(runDevice)
    vY = vY.to(runDevice)

    with torch.inference_mode():
        mZ = oModel.GetEmbedding(tX, vY)

    lEnc.append(mZ.cpu().numpy())
    lY.append(vY.cpu().numpy())

In [ ]:
# Latent Variables / Embeddings and Labels
mE = np.vstack(lEnc)
vY = np.hstack(lY)

In [ ]:
# Plot the Latent Variables / Embeddings Space
hA = PlotScatterData(mE, vY)
hA.set_title('Latent Variables / Embeddings Space');

* <font color='brown'>(**#**)</font> Intuitively, one may thing that the conditional data gives the model the ability to have a dedicated latent space per class. As there were a dedicated model per class.

In [ ]:
# Sample the Latent Spaces per Class

numGridPts = 7

vZ1 = torch.linspace(-3.0, 3.0, numGridPts, device = runDevice)
vZ2 = torch.linspace(-3.0, 3.0, numGridPts, device = runDevice)
mZ1, mZ2 = torch.meshgrid(vZ1, vZ2, indexing = 'ij')
mZ = torch.stack((mZ1.ravel(), mZ2.ravel()), dim = 1)

numSamples = mZ.shape[0]
lGrid      = []

with torch.inference_mode():
    for valY in L_CLASSES:
        vY = torch.full((numSamples,), valY, dtype = torch.long, device = runDevice)
        mY = oModel.oDecEmbedding(vY)
        tXHat = oModel.oDec(torch.cat((mZ, mY), dim = 1)).cpu()

        mGrid = torchvision.utils.make_grid(tXHat, nrow = numGridPts, pad_value = 0.5)
        lGrid.append(mGrid)

mGrid = torchvision.utils.make_grid(torch.stack(lGrid), nrow = 5, pad_value = 1.0)
mGrid = torch.clamp(mGrid.permute(1, 2, 0), 0.0, 1.0)

hF, hA = plt.subplots(figsize = (24, 10))
hA.imshow(mGrid, cmap = 'gray')
hA.axis('off');
# hF.suptitle('Conditional Latent Space Sampling per Class', fontsize = 16);

In [ ]:
# Class Conditional Linear Interpolation in Latent Space
numGridPts = 9
valY = 7

vX0 = np.array([-2.0, -2.0])
vX1 = np.array([ 2.0,  2.0])
vT = np.linspace(0, 1, numGridPts)

mZ = np.zeros((numGridPts, latDim), dtype = np.float32)
for ii in range(numGridPts):
    mZ[ii] = (1 - vT[ii]) * vX0 + vT[ii] * vX1

tZ = torch.tensor(mZ, device = runDevice)
vY = torch.full((numGridPts,), valY, dtype = torch.long, device = runDevice)

with torch.inference_mode():
    mY = oModel.oDecEmbedding(vY)
    tXHat = oModel.oDec(torch.cat((tZ, mY), dim = 1))

hF, vHa = plt.subplots(nrows = 1, ncols = numGridPts, figsize = (numGridPts * 1.5, 1.5))
vHa = vHa.flat

for ii in range(numGridPts):
    hA = vHa[ii]
    hA.imshow(TensorImageNumpy(tXHat[ii]), cmap = 'gray')
    hA.axis('off')

hF.suptitle(f'Class Conditional Interpolation, Label: {valY}', fontsize = 16);

* <font color='blue'>(**!**)</font> Fix ${\color{green}{\boldsymbol{z}}}$ and change the class label. Observe how the condition controls the generated digit.
* <font color='blue'>(**!**)</font> Change $\beta$ and retrain the model. Compare reconstruction quality with the latent space structure.
* <font color='green'>(**@**)</font> Create a grid which varies ${\color{green}{\boldsymbol{z}}}$ along one axis and the class label along the other axis.